# Stage 2: Feature Engineering & Exploratory Data Analysis (EDA)

**Project:** WD-GPR-SR (Battery State of Health Prediction with Uncertainty Quantification and Symbolic Regression)  
**Scope:** Feature extraction from charge/discharge cycles for NASA Ames (B0005, B0006, B0007, B0018) and CALCE (CS2_35, CS2_36, CS2_37, CS2_38).

### Objectives:
1. Derive operational thresholds dynamically per cell/dataset without hard-coding.
2. Extract robust health indicators: CCCT, CVCT, discharge plateau duration, temperature rises, and Incremental Capacity (IC) peak features.
3. Prevent data leakage: clearly separate inference-valid charge features from diagnostic post-discharge features.
4. Analyze feature correlations with SOH and quantify missing values.

In [1]:
# 1. Environment Setup & Configuration
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.config import DATASET_CONFIGS, FEATURES_MAIN, FEATURES_NASA_TEMP, FEATURES_DISCHARGE_DIAGNOSTIC
from src.feature_extraction import robust_savgol_smooth, compute_ic_curve, extract_all_features_for_cell

print('Configuration Loaded successfully.')
print('Inference-Valid Features (FEATURES_MAIN):', FEATURES_MAIN)
print('Derived NASA CC Current:', DATASET_CONFIGS['NASA']['cc_current_nominal'], 'A')
print('Derived CALCE CC Current:', DATASET_CONFIGS['CALCE']['cc_current_nominal'], 'A')

Configuration Loaded successfully.
Inference-Valid Features (FEATURES_MAIN): ['cycle', 'cc_charge_time', 'cv_charge_time', 'ic_peak_height', 'ic_peak_voltage']
Derived NASA CC Current: 1.51 A
Derived CALCE CC Current: 0.55 A


## 2. Smoothing & Incremental Capacity (IC) Peak Detection
The Savitzky-Golay helper enforces odd window lengths, caps window size at signal length, and guards against $dV \approx 0$ when computing $dQ/dV$.

In [3]:
# Demonstrate Savitzky-Golay smoothing and IC curve extraction on NASA B0005
with open('data/cache/nasa_B0005.pkl', 'rb') as f:
    b5_data = pickle.load(f)

ch = b5_data['time_series'][2]['charge']
t, v, i = ch['time'], ch['voltage'], ch['current']

pk_h, pk_v = compute_ic_curve(t, v, i, v_min=3.6, v_max=4.18)
print(f'NASA B0005 Cycle 2 IC Peak: Height = {pk_h:.4f} Ah/V at Voltage = {pk_v:.4f} V')

# Verify smoothing on a short and noisy signal
short_sig = np.array([3.6, 3.7, 3.75, 3.82, 3.9])
smoothed_short = robust_savgol_smooth(short_sig, window_length=15, polyorder=2)
print('Short signal smoothing length preserved:', len(smoothed_short) == len(short_sig))

NASA B0005 Cycle 2 IC Peak: Height = 5.5815 Ah/V at Voltage = 3.9894 V
Short signal smoothing length preserved: True


## 3. Extracting All Features Across Datasets
Load pre-cached cell objects, extract per-cycle feature dictionaries, and build clean DataFrames without fitting scalers (scaling belongs inside CV folds).

In [5]:
nasa_cells = ['B0005', 'B0006', 'B0007', 'B0018']
calce_cells = ['CS2_35', 'CS2_36', 'CS2_37', 'CS2_38']

nasa_dfs = []
for cell in nasa_cells:
    with open(f'data/cache/nasa_{cell}.pkl', 'rb') as f:
        cdata = pickle.load(f)
    cfg = DATASET_CONFIGS[cell]
    df_c = extract_all_features_for_cell(cell, cdata, cfg)
    nasa_dfs.append(df_c)
df_nasa = pd.concat(nasa_dfs, ignore_index=True)

calce_dfs = []
for cell in calce_cells:
    with open(f'data/cache/calce_{cell}.pkl', 'rb') as f:
        cdata = pickle.load(f)
    cfg = DATASET_CONFIGS[cell]
    df_c = extract_all_features_for_cell(cell, cdata, cfg)
    calce_dfs.append(df_c)
df_calce = pd.concat(calce_dfs, ignore_index=True)
df_all = pd.concat([df_nasa, df_calce], ignore_index=True)

print(f'NASA Features Table:  {df_nasa.shape}')
print(f'CALCE Features Table: {df_calce.shape}')
print(f'Unified Features:     {df_all.shape}')

NASA Features Table:  (636, 25)
CALCE Features Table: (4021, 25)
Unified Features:     (4657, 25)


## 4. Missing Value Analysis
CALCE does not log temperature signals; missing values are accurately logged as NaN rather than fabricated.

In [7]:
feat_cols = FEATURES_MAIN + FEATURES_NASA_TEMP + FEATURES_DISCHARGE_DIAGNOSTIC
nan_summary = pd.DataFrame({
    'Feature': feat_cols,
    'NASA_NaN': [df_nasa[c].isna().sum() for c in feat_cols],
    'NASA_Total': len(df_nasa),
    'CALCE_NaN': [df_calce[c].isna().sum() for c in feat_cols],
    'CALCE_Total': len(df_calce),
})
print(nan_summary.to_string(index=False))

                 Feature  NASA_NaN  NASA_Total  CALCE_NaN  CALCE_Total
                   cycle         0         636          0         4021
          cc_charge_time         0         636          0         4021
          cv_charge_time         0         636          0         4021
          ic_peak_height         5         636         14         4021
         ic_peak_voltage         5         636         14         4021
        temp_charge_mean         0         636       4021         4021
         temp_charge_max         0         636       4021         4021
        temp_charge_rise         0         636       4021         4021
          discharge_time         0         636          0         4021
voltage_plateau_duration         0         636          0         4021
  discharge_current_mean         0         636          0         4021
   discharge_current_max         0         636          0         4021
            voltage_mean         0         636          0         4021
      

## 5. Feature-vs-SOH Correlation Analysis
Evaluate Pearson correlation between each health indicator and SOH on valid (non-outlier) cycles.

In [9]:
valid_nasa = df_nasa[~df_nasa['is_outlier']]
valid_calce = df_calce[~df_calce['is_outlier']]

corr_nasa = {c: valid_nasa[c].corr(valid_nasa['SOH']) for c in feat_cols if valid_nasa[c].notna().sum() > 50}
corr_calce = {c: valid_calce[c].corr(valid_calce['SOH']) for c in feat_cols if valid_calce[c].notna().sum() > 50}

print('--- Top NASA Correlations with SOH ---')
for k, v in sorted(corr_nasa.items(), key=lambda x: abs(x[1]), reverse=True)[:6]:
    print(f'  {k:25s}: r = {v:+.4f}')

print('\n--- Top CALCE Correlations with SOH ---')
for k, v in sorted(corr_calce.items(), key=lambda x: abs(x[1]), reverse=True)[:6]:
    print(f'  {k:25s}: r = {v:+.4f}')

--- Top NASA Correlations with SOH ---
  voltage_plateau_duration : r = +0.9897
  voltage_slope            : r = +0.9649
  voltage_mean             : r = +0.9587
  discharge_time           : r = +0.9545
  cycle                    : r = -0.8942
  ic_peak_height           : r = +0.8560

--- Top CALCE Correlations with SOH ---
  discharge_time           : r = +0.9999
  voltage_plateau_duration : r = +0.9838
  cc_charge_time           : r = +0.9652
  ic_peak_voltage          : r = -0.9630
  voltage_mean             : r = +0.9412
  voltage_skew             : r = -0.9089


## 6. Summary of Feature Leakage Policy
The table in `reports/FEATURE_LEAKAGE_TABLE.md` documents the strict segregation between inference-valid features and diagnostic features.

In [11]:
print('Strictly Inference-Valid Features (FEATURES_MAIN):')
for i, f in enumerate(FEATURES_MAIN, 1):
    print(f'  {i}. {f} (Available prior to discharge)')
print('\nDischarge Features (Diagnostic / Post-Cycle Only - EXCLUDED from primary inference):')
for i, f in enumerate(FEATURES_DISCHARGE_DIAGNOSTIC, 1):
    print(f'  {i}. {f}')

Strictly Inference-Valid Features (FEATURES_MAIN):
  1. cycle (Available prior to discharge)
  2. cc_charge_time (Available prior to discharge)
  3. cv_charge_time (Available prior to discharge)
  4. ic_peak_height (Available prior to discharge)
  5. ic_peak_voltage (Available prior to discharge)

Discharge Features (Diagnostic / Post-Cycle Only - EXCLUDED from primary inference):
  1. discharge_time
  2. voltage_plateau_duration
  3. discharge_current_mean
  4. discharge_current_max
  5. voltage_mean
  6. voltage_std
  7. voltage_skew
  8. voltage_min
  9. voltage_slope
  10. temp_discharge_rise
